# Things I learned
algorithms, big O, timing code, optimizing, OOP, data cleaning

## Big O cheat sheet
- O(1) index lookup `L[0]`
- O(log n) binary search, halving loops
- O(n) one loop
- O(n log n) sorting
- O(n^2) nested loops
- O(2^n) all subsets (combinations)
- O(n!) all orderings (permutations)

## Binary search vs linear search - O(log n) vs O(n)

In [1]:
import time, random

def linear_search(L, target):
    for i, x in enumerate(L):
        if x == target:
            return i
    return None

def binary_search(sorted_list, target):
    low, high = 0, len(sorted_list) - 1
    while low <= high:
        mid = (low + high) // 2
        if sorted_list[mid] == target:
            return mid
        elif target < sorted_list[mid]:
            high = mid - 1      # throw away right half
        else:
            low = mid + 1       # throw away left half
    return None

for n in [10**5, 10**6, 10**7]:
    L = list(range(n))
    t = time.time(); linear_search(L, n - 1); t_lin = time.time() - t
    t = time.time(); binary_search(L, n - 1); t_bin = time.time() - t
    print(f"n={n:>9}  linear {t_lin:.5f}s   binary {t_bin:.7f}s")

n=   100000  linear 0.00346s   binary 0.0000072s
n=  1000000  linear 0.02486s   binary 0.0000052s


n= 10000000  linear 0.27860s   binary 0.0000079s


## Optimizing O(n^2) -> O(n) with sets
nested loop is quadratic, double n = 4x time. sets use hashing so lookup is O(1)

In [2]:
def find_common_elements(list1, list2):      # O(n^2)
    res = []
    for x in list1:
        for y in list2:
            if x == y:
                res.append(x)
    return res

def find_common_elements_fast(list1, list2): # O(n)
    return list(set(list1) & set(list2))

def find_missing_elements_fast(list1, list2):
    return list(set(list1) - set(list2))

for n in [1000, 2000, 4000]:
    list1 = random.choices(range(10*n), k=n)
    list2 = random.choices(range(10*n), k=n)
    t = time.time(); find_common_elements(list1, list2); slow = time.time() - t
    t = time.time(); find_common_elements_fast(list1, list2); fast = time.time() - t
    print(f"N={n}: nested {slow:.4f}s   set {fast:.6f}s")

N=1000: nested 0.0066s   set 0.000101s
N=2000: nested 0.0262s   set 0.000157s
N=4000: nested 0.1125s   set 0.000258s


## Counting pairs - O(n^2), only checks each pair once (j starts at i+1)

In [3]:
import math

def is_close(chair, other_chair):          # O(1)
    distance = math.sqrt((chair[0] - other_chair[0])**2 + (chair[1] - other_chair[1])**2)
    return distance <= 1.5

def has_close_seats(chair_list):           # O(n^2)
    for i in range(len(chair_list)):
        for j in range(i + 1, len(chair_list)):
            if is_close(chair_list[i], chair_list[j]):
                return True
    return False

print(has_close_seats([[0, 0], [5, 5], [5, 6]]))

True


## Exponential and factorial growth

In [4]:
import itertools

items = ['A', 'B', 'C', 'D']
subsets = [c for size in range(len(items) + 1) for c in itertools.combinations(items, size)]
print("subsets:", len(subsets), "= 2**4")
print("orderings:", len(list(itertools.permutations(items))), "= 4!")

subsets: 16 = 2**4
orderings: 24 = 4!


## timeit for measuring small code

In [5]:
import timeit
print(timeit.timeit('[0] * 1000', number=10000))
print(timeit.timeit('5000 in L', setup='L = list(range(10000))', number=1000))   # list: O(n)
print(timeit.timeit('5000 in S', setup='S = set(range(10000))', number=1000))    # set: O(1)

0.005897583003388718
0.028982167001231574
1.6582998796366155e-05


## Doomsday algorithm - weekday for any date with modular arithmetic

In [6]:
def day_of_week(day, month, year):
    month_codes = [0, 3, 3, 6, 1, 4, 6, 2, 5, 0, 3, 5]
    century = year // 100
    short_year = year % 100
    century_code = (2 * (3 - century % 4)) % 7
    calc = short_year + short_year // 4 + day + month_codes[month - 1] + century_code
    is_leap = (year % 4 == 0 and year % 100 != 0) or year % 400 == 0
    if is_leap and month <= 2:
        calc -= 1
    days = ["sunday", "monday", "tuesday", "wednesday", "thursday", "friday", "saturday"]
    return days[calc % 7]

print(day_of_week(24, 12, 1999))
print(day_of_week(29, 2, 2000))

friday
tuesday


## OOP - class attributes, static methods, getters/setters, state

In [7]:
class Library_Book:
    library_name = "City Library"        # class attribute, shared by all books

    @staticmethod
    def print_library_info():
        print("Welcome to the", Library_Book.library_name)

    def __init__(self, title, author, year):
        self.title = title
        self.author = author
        self.year = year
        self.times_borrowed = 0
        self.is_borrowed = False

    def getTitle(self):
        return self.title

    def changeTitle(self, title):
        self.title = title

    def compute_book_age(self):
        return 2026 - self.year

    def borrow(self):
        self.is_borrowed = True
        self.times_borrowed += 1

    def return_book(self):
        self.is_borrowed = False

book = Library_Book("Dune", "Frank Herbert", 1965)
Library_Book.print_library_info()
book.borrow(); book.return_book(); book.borrow()
print(book.getTitle(), book.compute_book_age(), book.times_borrowed, book.is_borrowed)

Welcome to the City Library
Dune 61 2 True


## Data cleaning with pandas - regex, typo mapping, median imputation

In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv("marketing_campaign_data_messy.csv")
df = df.drop(columns=['Clicks'])   # duplicate column
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df = df.drop_duplicates(subset='campaign_id', keep='first').reset_index(drop=True)

# "$1,200" -> 1200.0 with regex
df['spend'] = df['spend'].astype(str).str.replace(r'[^\d.-]', '', regex=True)
df['spend'] = pd.to_numeric(df['spend'], errors='coerce').abs()

# replace fake 500000 values with clicks * median cost per click
placeholder = df['spend'] == 500000
normal_cpc = (df.loc[~placeholder, 'spend'] / df.loc[~placeholder, 'clicks']).median()
df.loc[placeholder, 'spend'] = (df.loc[placeholder, 'clicks'] * normal_cpc).round(2)

df['channel'] = df['channel'].replace({'Facebok': 'Facebook', 'Insta_gram': 'Instagram',
    'Gogle': 'Google Ads', 'Tik_Tok': 'TikTok', 'E-mail': 'Email', 'N/A': np.nan}).fillna('Unknown')
df.head()

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24 00:00:00,2023-12-13,TikTok,16795,197,102.82,20.0,Y,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06 00:00:00,2023-05-12,Facebook,1860,30,24.33,1.0,0,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13 00:00:00,2023-12-20,Email,77820,843,1323.39,51.0,No,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22 00:00:00,2023-04-23,Facebook,7265,169,252.44,30.0,Yes,FA
